**PRC2 LoF + HomDel Calling Across TCGA**

This notebook contains loss-calling workflow for the PRC2 gene set: EZH2, EED, SUZ12, RBBP4, RBBP7


The workflow identifies:
- LoF mutations from GDC masked somatic mutation MAF files
- Homozygous deletions from GDC masked copy number segment files
- For CNV, masked segment files do not contain gene names, so gene-level
- HomDels are called by overlapping segment coordinates with gene coordinates validated against GENCODE Human Release 46 (GRCh38.p14) 

In [ ]:
# Load libraries

# Set path and use library
user_lib <- "/home/kennes38/ResearchProject/R/x86_64-pc-linux-gnu-library/4.5"
.libPaths(c(user_lib, .libPaths()))

library(TCGAbiolinks)
library(dplyr)
library(readr)
library(tibble)
library(purrr)
library(stringr)

In [ ]:
# Define paths, gene set and run mode

# Run from the project root on the HPC.
setwd("/home/kennes38/ResearchProject")

output_dir <- "20.1_prc2_LoFHomDel_finalcohort"
gdc_dir <- file.path(output_dir, "GDCdata")

dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)
dir.create(gdc_dir, showWarnings = FALSE, recursive = TRUE)


gene_set_label <- "prc2_loss"


target_genes <- c("EZH2", "EED", "SUZ12", "RBBP4", "RBBP7")


lof_classes <- c(
  "Nonsense_Mutation",
  "Frame_Shift_Del",
  "Frame_Shift_Ins",
  "Splice_Site",
  "Nonstop_Mutation",
  "Translation_Start_Site"
)

# Segment_Mean threshold for homozygous deletion calls.
homdel_threshold <- -1.5

# Turn debug messages on while testing. Set to FALSE for the full pancancer run.
debug <- FALSE

# 1 = test run, 2 = all TCGA projects.
# Run mode 1 first -> catch path/query issues before downloading many large GDC files.
run_mode <- 2

# Start with a small test
test_projects <- c("TCGA-LUSC")

all_projects <- TCGAbiolinks:::getGDCprojects()$project_id
all_tcga_projects <- grep("^TCGA-", all_projects, value = TRUE)

all_tcga_projects <- setdiff(
  all_tcga_projects,
  c("TCGA-CNTL", "TCGA-FPPP", "TCGA-LCML", "TCGA-MISC")
)

projects <- if (run_mode == 1) test_projects else all_tcga_projects

projects

In [ ]:
# Define GRCh38 Gene Coordinates for CNV Overlap (from GENCODE Human Release 46)

target_coords <- tibble::tribble(
  ~gene,   ~chr, ~start,     ~end,
  "EZH2",  "7",  148807257, 148884321,
  "EED",   "11", 86201212,   86278813,
  "SUZ12", "17", 31937007,   32001038,
  "RBBP4", "1",  32651142,   32686211,
  "RBBP7", "X",  16839283,   16870362
)

# Print the coordinates to record in notebook
target_coords


In [ ]:
# Define helper functions

debug_msg <- function(..., debug = TRUE) {
  # Small wrapper so all progress messages can be switched on/off from one flag.
  if (isTRUE(debug)) message(...)
}

standardise_tcga_ids <- function(tbl, barcode_col = "Tumor_Sample_Barcode") {
  # first 16 characters = sample-level ID used for RNA/TEdata matching
  # first 12 characters = patient-level ID
  tbl %>%
    mutate(
      Tumor_Sample_Barcode = as.character(.data[[barcode_col]]),
      sample_id_16 = substr(Tumor_Sample_Barcode, 1, 16),
      patient_id = substr(Tumor_Sample_Barcode, 1, 12)
    )
}

clean_chr <- function(x) {
  # This removes the chr prefix so chromosome labels compare correctly.
  gsub("^chr", "", as.character(x), ignore.case = TRUE)
}

n_distinct_if_present <- function(tbl, col) {
  # Function to avoid warnings such as 'Unknown or uninitialised column: sample_id_16'
  if (!col %in% colnames(tbl)) return(0L)
  dplyr::n_distinct(tbl[[col]])
}

In [ ]:
# Robust GDC file readers

read_maf_files_for_project <- function(query_maf, project, directory, debug = TRUE) {
  maf_meta <- getResults(query_maf)

  debug_msg("MAF metadata rows: ", nrow(maf_meta), debug = debug)
  debug_msg("First metadata file names:", debug = debug)
  if (isTRUE(debug)) print(head(maf_meta$file_name))

  GDCdownload(
    query_maf,
    method = "api",
    directory = directory,
    files.per.chunk = 1
  )

  all_files <- list.files(directory, recursive = TRUE, full.names = TRUE)
  maf_files <- all_files[basename(all_files) %in% maf_meta$file_name]

  debug_msg("MAF files found on disk: ", length(maf_files), debug = debug)
  if (isTRUE(debug)) print(head(basename(maf_files)))

  missing_files <- setdiff(maf_meta$file_name, basename(maf_files))
  if (length(missing_files) > 0) {
    warning(
      "Some MAF files listed in metadata were not found on disk for ",
      project,
      ". Missing files: ",
      paste(head(missing_files, 10), collapse = "; "),
      ifelse(length(missing_files) > 10, " ...", "")
    )
  }

  if (length(maf_files) == 0) {
    warning("No downloaded MAF files matched metadata file_name values for ", project)
    return(tibble())
  }

  maf_df <- purrr::map_dfr(maf_files, function(f) {
    fname <- basename(f)
    meta_row <- maf_meta %>% filter(file_name == fname)

    df <- tryCatch(
      readr::read_tsv(
        f,
        comment = "#",
        # Reading all columns as character avoids bind_rows/map_dfr type-combination failures
        col_types = readr::cols(.default = readr::col_character()),
        show_col_types = FALSE,
        progress = FALSE
      ),
      error = function(e) {
        warning("Failed to read MAF file ", fname, ": ", conditionMessage(e))
        return(tibble())
      }
    )

    if (nrow(df) == 0) return(tibble())

    df %>%
      mutate(
        project = project,
        maf_file_name = fname,
        maf_file_id = meta_row$file_id[1]
      )
  })

  debug_msg("MAF rows read: ", nrow(maf_df), debug = debug)

  maf_df
}

read_cnv_segments_for_project <- function(query_cnv, project, directory, debug = TRUE) {
  # Function to read each downloaded TSV manually
  # Uses the query metadata to recover the correct TCGA barcode for each file
  cnv_meta <- getResults(query_cnv)

  debug_msg("CNV metadata rows: ", nrow(cnv_meta), debug = debug)
  debug_msg("First metadata file names:", debug = debug)
  if (isTRUE(debug)) print(head(cnv_meta$file_name))

  # Download files
  GDCdownload(
    query_cnv,
    method = "api",
    directory = directory,
    files.per.chunk = 1
  )

  all_files <- list.files(directory, recursive = TRUE, full.names = TRUE)
  cnv_files <- all_files[basename(all_files) %in% cnv_meta$file_name]

  debug_msg("CNV files found on disk: ", length(cnv_files), debug = debug)
  if (isTRUE(debug)) print(head(basename(cnv_files)))

  if (length(cnv_files) == 0) {
    warning("No downloaded CNV files matched metadata file_name values for ", project)
    return(tibble())
  }

  cnv_df <- purrr::map_dfr(cnv_files, function(f) {
    fname <- basename(f)
    meta_row <- cnv_meta %>% filter(file_name == fname)

    if (nrow(meta_row) == 0) {
      warning("No metadata row for CNV file: ", fname)
      return(tibble())
    }

    if (nrow(meta_row) > 1) {
      warning("Multiple metadata rows for CNV file: ", fname, "; using the first row")
      meta_row <- meta_row[1, ]
    }

    sample_barcode <- meta_row$sample.submitter_id[1]
    case_barcode <- meta_row$cases.submitter_id[1]

    if (is.na(sample_barcode) || sample_barcode == "") {
      sample_barcode <- case_barcode
    }

    df <- tryCatch(
      readr::read_tsv(f, show_col_types = FALSE, progress = FALSE),
      error = function(e) {
        warning("Failed to read CNV file ", fname, ": ", conditionMessage(e))
        return(tibble())
      }
    )

    if (nrow(df) == 0) return(tibble())

    df %>%
      mutate(
        project = project,
        cnv_file_name = fname,
        cnv_file_id = meta_row$file_id[1],
        cnv_metadata_id = meta_row$id[1],
        Tumor_Sample_Barcode = sample_barcode,
        cases_submitter_id = case_barcode
      )
  })

  debug_msg("CNV rows read: ", nrow(cnv_df), debug = debug)
  debug_msg("Mapped CNV sample barcodes: ", n_distinct(cnv_df$Tumor_Sample_Barcode), debug = debug)

  cnv_df
}

In [ ]:
# Main project function

run_gene_set_project <- function(project) {
  # Function to run full loss-calling workflow for one TCGA project
  # Returns one summary row and writes three per-project CSVs
  message("===== ", project, " =====")

  mutated_samples <- tibble()
  homdel_samples <- tibble()

  # LoF mutation calls are already gene-level
  # Filter the MAF to target genes and LoF variant classes
  query_maf <- tryCatch(
    GDCquery(
      project = project,
      data.category = "Simple Nucleotide Variation",
      data.type = "Masked Somatic Mutation"
    ),
    error = function(e) {
      warning("MAF query failed for ", project, ": ", conditionMessage(e))
      return(NULL)
    }
  )

  if (!is.null(query_maf)) {
    tryCatch({
      maf <- read_maf_files_for_project(
        query_maf = query_maf,
        project = project,
        directory = gdc_dir,
        debug = debug
      )

      if (nrow(maf) > 0) {
        required_maf_cols <- c("Hugo_Symbol", "Variant_Classification", "Tumor_Sample_Barcode")
        missing_maf_cols <- setdiff(required_maf_cols, colnames(maf))

        if (length(missing_maf_cols) > 0) {
          warning(
            "MAF table for ",
            project,
            " is missing required columns: ",
            paste(missing_maf_cols, collapse = ", ")
          )
        } else {
          mutated_samples <- maf %>%
            filter(
              Hugo_Symbol %in% target_genes,
              Variant_Classification %in% lof_classes
            ) %>%
            transmute(
              project = project,
              gene_set = gene_set_label,
              gene = Hugo_Symbol,
              event = "LoF",
              mutation_class = Variant_Classification,
              Tumor_Sample_Barcode,
              sample_id_16 = substr(Tumor_Sample_Barcode, 1, 16),
              patient_id = substr(Tumor_Sample_Barcode, 1, 12)
            ) %>%
            distinct()
        }
      }

      debug_msg("LoF rows: ", nrow(mutated_samples), debug = debug)
      debug_msg("LoF samples: ", n_distinct_if_present(mutated_samples, "sample_id_16"), debug = debug)
    }, error = function(e) {
      warning("MAF processing failed for ", project, ": ", conditionMessage(e))
    })
  }

  # CNV calls are segment-level
  # Script overlaps each segment with the GRCh38 gene coordinates above.
  query_cnv <- tryCatch(
    GDCquery(
      project = project,
      data.category = "Copy Number Variation",
      data.type = "Masked Copy Number Segment"
    ),
    error = function(e) {
      warning("CNV query failed for ", project, ": ", conditionMessage(e))
      return(NULL)
    }
  )

  if (!is.null(query_cnv)) {
    tryCatch({
      cnv_df <- read_cnv_segments_for_project(
        query_cnv = query_cnv,
        project = project,
        directory = gdc_dir,
        debug = debug
      )

      if (nrow(cnv_df) > 0) {
        cnv_df <- cnv_df %>%
          mutate(
            chr_clean = clean_chr(Chromosome),
            Start = as.numeric(Start),
            End = as.numeric(End),
            Segment_Mean = as.numeric(Segment_Mean)
          )

        gene_segment_overlaps <- purrr::map_dfr(seq_len(nrow(target_coords)), function(i) {
          g <- target_coords[i, ]

          # Segment overlaps a gene unless ends before the gene starts / starts after the gene ends.
          cnv_df %>%
            filter(chr_clean == g$chr) %>%
            filter(!(End < g$start | Start > g$end)) %>%
            mutate(gene = g$gene)
        })

        debug_msg("Gene-segment overlaps before threshold: ", nrow(gene_segment_overlaps), debug = debug)

        homdel_samples <- gene_segment_overlaps %>%
          # Apply homozygous deletion threshold.
          filter(Segment_Mean <= homdel_threshold) %>%
          transmute(
            project = project,
            gene_set = gene_set_label,
            gene,
            event = "HomDel",
            mutation_class = NA_character_,
            Tumor_Sample_Barcode,
            sample_id_16 = substr(Tumor_Sample_Barcode, 1, 16),
            patient_id = substr(Tumor_Sample_Barcode, 1, 12),
            Segment_Mean,
            Chromosome,
            Start,
            End,
            cnv_file_name,
            cnv_file_id
          ) %>%
          distinct()

        debug_msg("HomDel rows after threshold: ", nrow(homdel_samples), debug = debug)
        debug_msg("HomDel samples: ", n_distinct(homdel_samples$sample_id_16), debug = debug)
      }
    }, error = function(e) {
      warning("CNV processing failed for ", project, ": ", conditionMessage(e))
    })
  }

  
  # Combine and save per-project outputs
  all_loss <- bind_rows(
    mutated_samples,
    homdel_samples %>%
      select(project, gene_set, gene, event, mutation_class, Tumor_Sample_Barcode, sample_id_16, patient_id)
  ) %>%
    distinct()

  write_csv(mutated_samples, file.path(output_dir, paste0(project, "_", gene_set_label, "_LoF_samples.csv")))
  write_csv(homdel_samples, file.path(output_dir, paste0(project, "_", gene_set_label, "_HomDel_samples.csv")))
  write_csv(all_loss, file.path(output_dir, paste0(project, "_", gene_set_label, "_all_loss.csv")))

  tibble(
    project = project,
    gene_set = gene_set_label,
    n_lof = n_distinct_if_present(mutated_samples, "sample_id_16"),
    n_homdel = n_distinct_if_present(homdel_samples, "sample_id_16"),
    n_any_loss = n_distinct_if_present(all_loss, "sample_id_16")
  )
}

In [ ]:
# Run project list

summary_tbl <- purrr::map_dfr(projects, run_gene_set_project)

summary_tbl

write_csv(
  summary_tbl,
  file.path(output_dir, paste0("pancancer_", gene_set_label, "_summary.csv"))
)

In [ ]:
completed_files <- list.files(
  output_dir,
  pattern = paste0("_", gene_set_label, "_all_loss\\.csv$"),
  full.names = FALSE
)

completed_projects <- sub(
  paste0("_", gene_set_label, "_all_loss\\.csv$"),
  "",
  completed_files
)

projects_to_run <- setdiff(projects, completed_projects)

cat("Projects requested:", length(projects), "\n")
cat("Projects already complete:", length(intersect(projects, completed_projects)), "\n")
cat("Projects still to run:", length(projects_to_run), "\n")
print(projects_to_run)

# Run only unfinished projects
if (length(projects_to_run) > 0) {
  purrr::walk(projects_to_run, run_gene_set_project)
} else {
  message("All requested projects are already complete.")
}

count_saved_samples <- function(project, suffix) {
  path <- file.path(
    output_dir,
    paste0(project, "_", gene_set_label, suffix)
  )

  if (!file.exists(path)) return(NA_integer_)

  saved <- readr::read_csv(path, show_col_types = FALSE)
  if (!"sample_id_16" %in% names(saved)) return(0L)
  dplyr::n_distinct(saved$sample_id_16, na.rm = TRUE)
}

summary_tbl <- purrr::map_dfr(projects, function(project) {
  tibble(
    project = project,
    gene_set = gene_set_label,
    n_lof = count_saved_samples(project, "_LoF_samples.csv"),
    n_homdel = count_saved_samples(project, "_HomDel_samples.csv"),
    n_any_loss = count_saved_samples(project, "_all_loss.csv")
  )
})

write_csv(
  summary_tbl,
  file.path(output_dir, paste0("pancancer_", gene_set_label, "_summary.csv"))
)

summary_tbl

In [ ]:
# Build pancancer event and patients table

all_loss_files <- list.files(
  output_dir,
  pattern = paste0("_", gene_set_label, "_all_loss\\.csv$"),
  full.names = TRUE
)

loss_events <- purrr::map_dfr(all_loss_files, read_csv, show_col_types = FALSE) %>%
  distinct()

loss_patients <- loss_events %>%
  # File used for RNA/TEdata matching and WT-control matching.
  group_by(project, gene_set, sample_id_16, patient_id) %>%
  summarise(
    genes_lost = paste(sort(unique(gene)), collapse = ";"),
    events = paste(sort(unique(event)), collapse = ";"),
    lof_genes = paste(sort(unique(gene[event == "LoF"])), collapse = ";"),
    homdel_genes = paste(sort(unique(gene[event == "HomDel"])), collapse = ";"),
    mutation_classes = paste(sort(unique(na.omit(mutation_class))), collapse = ";"),
    n_genes_lost = n_distinct(gene),
    n_loss_events = n(),
    has_lof = any(event == "LoF"),
    has_homdel = any(event == "HomDel"),
    .groups = "drop"
  )

write_csv(
  loss_events,
  file.path(output_dir, paste0("pancancer_", gene_set_label, "_events.csv"))
)

write_csv(
  loss_patients,
  file.path(output_dir, paste0("pancancer_", gene_set_label, "_patients.csv"))
)

cat("Events:", nrow(loss_events), "\n")
cat("Unique samples:", n_distinct(loss_events$sample_id_16), "\n")
cat("Unique patients:", n_distinct(loss_events$patient_id), "\n")

loss_patients %>%
  count(project, sort = TRUE)